# Synthetic demonstration - statin timing and diabetic eye disease

**This notebook uses fully synthetic, randomly generated data. It contains no real patient records.**

Its only purpose is to let anyone without UK Biobank or CPRD access run the core analysis logic end to end and see how the timing of statin initiation relates to diabetic eye disease (DED) in this project.

The real analyses live in `code/` and require approved data access. Numbers produced below are meaningless (the data are random); only the *method* is illustrative.

Pipeline shown here:
1. Generate a synthetic cohort
2. Define the exposure (`days_from_diabetes_to_statins`) and outcome (`diabetic_eye_disease`)
3. Adjusted logistic regression (timing + covariates)
4. Propensity score matching (PSM) and Inverse Probability of Treatment Weighting (IPTW) for early vs later initiation, adjusting for the same covariates

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

RNG = np.random.default_rng(42)  # reproducible synthetic data
N = 5000

## 1. Generate a synthetic cohort

Covariates mirror the *structure* of the real analysis (sex, smoking, alcohol, diabetes type, deprivation, BMI) but are drawn at random. We build in a mild positive association between later initiation and DED so the demo is illustrative; this is an artefact of the simulation, not a finding.

In [ ]:
df = pd.DataFrame({
    'sex': RNG.integers(0, 2, N),                       # 0 = female, 1 = male
    'smoking_status': RNG.integers(0, 3, N),            # 0 never, 1 former, 2 current
    'alcohol_intake_frequency': RNG.integers(0, 4, N),
    'diabetes_type': RNG.integers(1, 3, N),             # 1 = type 1, 2 = type 2
    'townsend_deprivation_index': RNG.normal(0, 3, N),
    'bmi': RNG.normal(28, 5, N).clip(15, 55),
})

# Exposure: days from diabetes diagnosis to statin initiation.
# Make timing depend on covariates so that early/later groups differ at baseline
# (i.e. genuine confounding for the propensity model to adjust for).
base_days = (RNG.gamma(shape=2.0, scale=180, size=N)
             + 40 * df['sex']
             + 30 * (df['diabetes_type'] == 2)
             + 8 * df['townsend_deprivation_index'])
df['days_from_diabetes_to_statins'] = base_days.clip(0).round().astype(int)

# Grouped exposure: early (<=180 days) vs later (>180 days)
df['early_statin'] = (df['days_from_diabetes_to_statins'] <= 180).astype(int)

# Synthetic outcome: mild positive effect of delay + covariate effects + noise
lin = (-2.0
       + 0.0009 * df['days_from_diabetes_to_statins']
       + 0.25 * df['sex']
       + 0.20 * (df['smoking_status'] == 1)
       - 0.30 * (df['diabetes_type'] == 2)
       + 0.03 * df['townsend_deprivation_index'])
prob = 1 / (1 + np.exp(-lin))
df['diabetic_eye_disease'] = (RNG.random(N) < prob).astype(int)

print(df.shape)
df.head()

## 2. Adjusted logistic regression (the main timing-DED analysis)

DED regressed on statin timing, adjusting for sex, smoking, alcohol, diabetes type, deprivation and BMI. In the real data, later initiation was associated with higher odds of DED after adjustment; here the coefficient is whatever the synthetic data happen to produce.

In [ ]:
covariates = ['sex', 'smoking_status', 'alcohol_intake_frequency',
              'diabetes_type', 'townsend_deprivation_index', 'bmi']

reg_vars = ['days_from_diabetes_to_statins'] + covariates
X = sm.add_constant(df[reg_vars].astype(float))
y = df['diabetic_eye_disease']

model = sm.Logit(y, X).fit(disp=False)

odds_ratios = np.exp(model.params)
summary = pd.DataFrame({'OR': odds_ratios, 'p_value': model.pvalues})
print('OR per additional YEAR of delay:',
      round(np.exp(model.params['days_from_diabetes_to_statins'] * 365), 3))
summary.round(4)

## 3. PSM and IPTW (early vs later initiation)

The propensity score models the probability of **early initiation given the covariates** (sex, smoking, alcohol, diabetes type, deprivation and BMI). Crucially, the exposure timing itself is *not* used as a predictor here; the propensity model exists to balance the confounders between the early and later groups. The score is then used to (a) 1:1 nearest-neighbour match and (b) form IPTW weights, before comparing DED rates.

These are complementary, exploratory estimates alongside the adjusted regression above.

In [ ]:
# Propensity score: P(early initiation | covariates)
treat = df['early_statin'].values
Xps = df[covariates].astype(float).values

ps = LogisticRegression(max_iter=1000).fit(Xps, treat).predict_proba(Xps)[:, 1]
df['propensity'] = ps
y = df['diabetic_eye_disease'].values

# IPTW 
w = np.where(treat == 1, 1 / ps, 1 / (1 - ps))
rate_treated = np.average(y[treat == 1], weights=w[treat == 1])
rate_control = np.average(y[treat == 0], weights=w[treat == 0])
print(f'IPTW  DED rate  early={rate_treated:.3f}  later={rate_control:.3f}  ATE={rate_treated - rate_control:+.3f}')

# 1:1 nearest-neighbour PSM on the propensity score 
idx_t = np.where(treat == 1)[0]
idx_c = np.where(treat == 0)[0]
nn = NearestNeighbors(n_neighbors=1).fit(ps[idx_c].reshape(-1, 1))
_, match = nn.kneighbors(ps[idx_t].reshape(-1, 1))
matched_c = idx_c[match.flatten()]
ate_psm = y[idx_t].mean() - y[matched_c].mean()
print(f'PSM   DED rate  early={y[idx_t].mean():.3f}  later={y[matched_c].mean():.3f}  ATE={ate_psm:+.3f}')

## Notes

- All numbers above come from random data seeded with `np.random.default_rng(42)` and they are reproducible but scientifically meaningless.
- The real pipeline (in `code/`) applies this logic to UK Biobank and CPRD, with proper phenotyping via the published codelists (see the main README).
- As set out in the README, the causal estimates in the real study are association-level. Further analysis such as a target trial emulation would be the more rigorous approach to the timing question.